# MetaTrader 5 - EURUSD Tick Chart Analysis

This notebook demonstrates how to connect to the **MetaTrader 5 (MT5)** trading terminal using its Python integration, fetch tick data for a specific financial symbol (e.g., `EURUSD`), perform basic data processing, and visualize a tick chart.

### Features:
1. Initialize connection to MT5 terminal.
2. Query tick range for a specific day (default: yesterday).
3. Preprocess tick data into a pandas DataFrame.
4. Calculate bid-ask spreads using a **universal formula** based on instrument details (`mt5.symbol_info`).
5. Visualize the full-day tick prices and spread, along with a detailed zoomed-in view of step-wise tick movement.

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

# Set pandas options for better display
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

In [ ]:
# Initialize MetaTrader 5 connection
if not mt5.initialize():
    print("MetaTrader5 initialization failed, error code:", mt5.last_error())
    quit()
else:
    print("MetaTrader5 initialized successfully.")

    # Print terminal connection info
    info = mt5.terminal_info()
    print(f"Connected to: {info.company} - {info.name}")
    print(f"Version: {mt5.version()}")

## Define Parameters

By default, we fetch ticks for the `EURUSD` symbol for **yesterday** (the day prior to current execution).
You can customize the `SYMBOL` and date range below.

In [ ]:
# --- Configuration ---
SYMBOL = "EURUSD"  # Symbol to fetch
today = datetime.now()
yesterday = today - timedelta(days=4)

# Start and end of yesterday
date_from = datetime(yesterday.year, yesterday.month, yesterday.day, 0, 0, 0)
date_to = datetime(yesterday.year, yesterday.month, yesterday.day, 23, 59, 59)

print(f"Symbol: {SYMBOL}")
print(f"Date From: {date_from}")
print(f"Date To:   {date_to}")

## Fetch Tick Data

We use `mt5.copy_ticks_range` to request tick data. The flag `mt5.COPY_TICKS_ALL` retrieves all tick types (bid, ask, etc.).

In [ ]:
print(f"Requesting tick data for {SYMBOL}...")
ticks = mt5.copy_ticks_range(SYMBOL, date_from, date_to, mt5.COPY_TICKS_ALL)

if ticks is None or len(ticks) == 0:
    print(f"No ticks retrieved. Check if {SYMBOL} is available in Market Watch or if the market was open during the selected range.")
    print("Error code:", mt5.last_error())
else:
    print(f"Successfully retrieved {len(ticks):,} ticks.")

## Preprocess Data

Let's convert the fetched ticks array into a pandas DataFrame, format the timestamps, and compute the raw spread (`ask - bid`).

To make the calculation universal, we fetch the symbol's information using `mt5.symbol_info` to get its `point` size and `digits` count. A standard **pip** is typically defined as 10 points for currency pairs with 3 or 5 decimal digits, and as 1 point for other symbols (such as gold, crypto, or indices).

In [ ]:
if ticks is not None and len(ticks) > 0:
    # Create DataFrame
    df = pd.DataFrame(ticks)

    # Convert millisecond timestamp to pandas datetime
    df['time'] = pd.to_datetime(df['time_msc'], unit='ms')

    # Set the time column as the index for easier analysis
    df.set_index('time', inplace=True)

    # Fetch instrument info for universal pip calculation
    info = mt5.symbol_info(SYMBOL)
    if info is not None:
        point = info.point
        # A standard pip is 10 points for 3/5 digit forex pairs, and 1 point for others
        if info.digits in [3, 5]:
            pip_size = 10 * point
        else:
            pip_size = point
        print(f"Universal Pip Calculation: 1 Pip = {pip_size} (Point: {point}, Digits: {info.digits})")
    else:
        pip_size = 0.0001
        print(f"Symbol info not found for {SYMBOL}. Using fallback 1 Pip = {pip_size}")

    # Calculate bid-ask spread
    df['spread_raw'] = df['ask'] - df['bid']
    df['spread_pips'] = df['spread_raw'] / pip_size

    # Display the first few rows
    print("\nPreprocessed Tick DataFrame:")
    display(df.head())
else:
    print("No data to preprocess.")

## Summary Statistics

Let's look at the basic statistics of the ticks, such as average, maximum, and minimum spreads.

In [ ]:
if ticks is not None and len(ticks) > 0:
    print("--- Summary Statistics ---")
    print(f"Total Ticks: {len(df):,}")
    print(f"Min Bid: {df['bid'].min():.5f}")
    print(f"Max Ask: {df['ask'].max():.5f}")
    print(f"Average Spread (pips): {df['spread_pips'].mean():.2f}")
    print(f"Max Spread (pips): {df['spread_pips'].max():.2f}")
    print(f"Min Spread (pips): {df['spread_pips'].min():.2f}")
else:
    print("No data available for statistics.")

## Visualizing the Tick Chart

We construct a multi-panel visual display:
1. **Full-Day price action** using a simple line plot of Bid/Ask.
2. **Full-Day spread** over time (in pips).
3. **Detailed Zoom-in View** of 200 ticks from the middle of the day. A step plot (`where='post'`) is used here because price doesn't change smoothly between ticks; it steps discretely.

In [ ]:
if ticks is not None and len(ticks) > 0:
    # Set up matplotlib figure
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 16))

    # Plot 1: Full-Day Bid/Ask Price Action
    ax1.plot(df.index, df['bid'], label='Bid', color='blue', alpha=0.5)
    ax1.plot(df.index, df['ask'], label='Ask', color='red', alpha=0.5)
    ax1.set_title(f"{SYMBOL} Tick Prices (Full Day) - {yesterday.strftime('%Y-%m-%d')}")
    ax1.set_ylabel("Price")
    ax1.set_xlabel("Time")
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Plot 2: Full-Day Spread (Pips)
    ax2.plot(df.index, df['spread_pips'], label='Spread (pips)', color='green', alpha=0.6)
    ax2.set_ylabel("Spread (Pips)")
    ax2.set_xlabel("Time")
    ax2.set_title(f"{SYMBOL} Bid-Ask Spread (Full Day)")
    ax2.legend(loc='upper left')
    ax2.grid(True, linestyle='--', alpha=0.5)

    # Plot 3: Zoomed view of 200 ticks in the middle of the day
    mid_idx = len(df) // 2
    zoom_df = df.iloc[mid_idx:mid_idx + 200]

    ax3.step(zoom_df.index, zoom_df['bid'], label='Bid', color='blue', where='post', marker='.')
    ax3.step(zoom_df.index, zoom_df['ask'], label='Ask', color='red', where='post', marker='.')
    ax3.set_title(f"{SYMBOL} Detailed Step View (200 Ticks starting from {zoom_df.index[0].strftime('%H:%M:%S')})")
    ax3.set_ylabel("Price")
    ax3.set_xlabel("Time (HH:MM:SS.mmm)")
    ax3.legend(loc='upper left')
    ax3.grid(True, linestyle='--', alpha=0.5)

    # Format detailed time display with microseconds/milliseconds on x-axis
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f'))

    # Rotate date labels slightly for readability on all subplots individually
    plt.setp(ax1.get_xticklabels(), rotation=15, ha='right')
    plt.setp(ax2.get_xticklabels(), rotation=15, ha='right')
    plt.setp(ax3.get_xticklabels(), rotation=15, ha='right')

    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

## Shutdown Connection

Always close the connection to the MetaTrader 5 terminal after completing data tasks.

In [ ]:
# Shutdown the connection
mt5.shutdown()
print("MetaTrader5 connection closed.")